# R21 Resting-State Randomise Results

Inspect every N=27 **cluster-extent corrected** primary and secondary randomise result on MNI anatomy, summarize the corresponding dual-regression stage-2 beta for sham, RTPJ, VLPFC, and BOTH, and review the condition-level MRIQC boxplots used to evaluate differential motion. Each result is shown in its own interactive NiiVue viewer. TFCE results are intentionally excluded.

## 1. Verify notebook dependencies

Launch this notebook with `bash notebooks/run_randomise_notebook.sh`. IPyNiiVue uses both a Python kernel package and an AnyWidget browser extension, so installing it after JupyterLab has already started is not sufficient.

In [ ]:
from pathlib import Path
import importlib.util

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'code' / 'check_randomise_results.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the r21-rest repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
required_imports = ('ipyniivue', 'ipywidgets', 'matplotlib', 'nibabel', 'nilearn', 'numpy', 'pandas')
missing = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Close JupyterLab and run: bash notebooks/run_randomise_notebook.sh')
print('Notebook dependencies and IPyNiiVue kernel package are available.')
print('Project root:', PROJECT_ROOT)

## 2. Load significant cluster-extent results

FSL corrected-p images contain `1-p`; the notebook defines the displayed region as voxels with `1-p > 0.95`. The two tracked summary tables cover 77 primary and 98 secondary network-by-contrast jobs. Each network/contrast is interpreted as an individual hypothesis and all results are reported. Following Rubin's inference-based framework, no blanket alpha adjustment is applied merely because 175 jobs were run; an adjustment would be relevant only for a disjunctive claim that at least one effect exists somewhere in that collection.

In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, image
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from ipyniivue import NiiVue

CORRP_THRESHOLD = 0.95
CONDITION_ORDER = ['sham', 'rtpj', 'vlpfc', 'both']
CONDITION_LABELS = ['Sham', 'RTPJ', 'VLPFC', 'BOTH']
NETWORK_LABELS = {
    'dmn': 'default mode network (DMN)',
    'ecn': 'executive control network (ECN)',
    'right-fpn': 'right frontoparietal network',
    'left-fpn': 'left frontoparietal network',
    'primary-visual': 'primary visual network',
    'occipital-pole': 'occipital-pole visual network',
    'lateral-visual': 'lateral visual network',
    'primary-visual-lateral-visual': 'combined primary/lateral visual component',
    'sensorimotor': 'sensorimotor network',
    'auditory': 'auditory network',
}
ANALYSIS_LABELS = {'0': 'data-derived ICA, automatic dimensionality', '20': 'data-derived ICA, 20 components', 'smith09': 'direct Smith09 RSN map'}
SUMMARY_DIR = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary'
SUMMARY_SPECS = [
    ('primary', SUMMARY_DIR / 'task-rest_randomise_peak_summary.tsv'),
    ('secondary', SUMMARY_DIR / 'task-rest_desc-SecondaryNetworks_randomise_peak_summary.tsv'),
]
missing_summaries = [str(path) for _, path in SUMMARY_SPECS if not path.is_file()]
if missing_summaries:
    raise FileNotFoundError('Missing randomise summaries: ' + ', '.join(missing_summaries))

result_tables = []
for family, path in SUMMARY_SPECS:
    table = pd.read_csv(path, sep='\t', dtype=str).fillna('')
    table['family'] = family
    table['summary_file'] = str(path.relative_to(PROJECT_ROOT))
    result_tables.append(table)
results = pd.concat(result_tables, ignore_index=True)

if 'roi_values_tsv' not in results.columns:
    raise RuntimeError('Rerun code/check_randomise_results.py on the Linux box and push the generated ROI-value TSVs.')
significant = results.loc[(results['inference'] == 'cluster-extent') & (results['peak_gt_threshold'] == 'true') & (results['status'] == 'ok')].copy()
contrast_terms = {
    'both-minus-sham': ('BOTH', 'sham'),
    'both-minus-rtpj': ('BOTH', 'RTPJ'),
    'both-minus-vlpfc': ('BOTH', 'VLPFC'),
    'rtpj-minus-vlpfc': ('RTPJ', 'VLPFC'),
    'rtpj-minus-sham': ('RTPJ', 'sham'),
    'vlpfc-minus-sham': ('VLPFC', 'sham'),
    'both-minus-mean-rtpj-vlpfc': ('BOTH', 'average(RTPJ, VLPFC)'),
}
def signed_effect(row):
    first, second = contrast_terms[row['condition_contrast']]
    if row['direction'] == 'negative':
        first, second = second, first
    return f'{first} > {second}'

significant['effect'] = significant.apply(signed_effect, axis=1)
significant['peak_fwe_p'] = 1.0 - significant['peak_corrp'].astype(float)
significant['family'] = pd.Categorical(significant['family'], ['primary', 'secondary'], ordered=True)
significant = significant.sort_values(['family', 'peak_fwe_p', 'analysis', 'network']).reset_index(drop=True)
if significant.empty:
    raise RuntimeError('The summary contains no complete cluster-extent maps with peak 1-p > 0.95.')
if (significant['roi_values_tsv'].str.strip() == '').any():
    raise RuntimeError('Portable ROI values are missing. Rerun code/check_randomise_results.py on Linux, then commit and push derivatives/fsl/randomise_summary.')
significant['analysis_description'] = significant['analysis'].map(ANALYSIS_LABELS)
significant['network_description'] = significant['network'].map(NETWORK_LABELS).fillna(significant['network'])
summary_table = significant[['family', 'analysis_description', 'network_description', 'component', 'effect', 'peak_fwe_p']].copy()
summary_table.columns = ['Family', 'Analysis', 'Network', 'Map/component', 'Signed effect', 'Peak FWE-corrected p']
summary_table['Family'] = summary_table['Family'].astype(str).str.title()
summary_table.insert(0, 'Result', range(1, len(summary_table) + 1))
primary_count = int((significant['family'] == 'primary').sum())
secondary_count = int((significant['family'] == 'secondary').sum())
print(f'Cluster-extent results available: {len(significant)} ({primary_count} primary; {secondary_count} secondary)')
print('Across-job interpretation follows separate individual hypotheses; no global alpha adjustment is applied.')
display(summary_table.style.format({'Peak FWE-corrected p': '{:.4f}'}).hide(axis='index'))

## 3. Define visualization and ROI-summary helpers

In [ ]:
def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

def result_image(row):
    copied = str(row.get('copied_image', '')).strip()
    source = copied or str(row['corrp_file']).strip()
    path = project_path(source)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

def result_label(row):
    return f"{row['effect']} in the {row['network_description']}"

def result_sidecar(row):
    path = project_path(row['copied_sidecar'])
    return json.loads(path.read_text()) if path.is_file() else {}

def extract_condition_values(row_index):
    row = significant.loc[row_index]
    corrp_img = nib.load(str(result_image(row)))
    corrp_data = np.asarray(corrp_img.dataobj)
    roi = np.isfinite(corrp_data) & (corrp_data > CORRP_THRESHOLD)
    if not roi.any():
        raise ValueError('Selected corrected-p map has no voxels above the threshold.')

    values_path = project_path(row['roi_values_tsv'])
    if not values_path.is_file():
        raise FileNotFoundError(f'Portable ROI-value table is not tracked locally: {values_path}')
    values = pd.read_csv(values_path, sep='\t', dtype={'participant': str, 'condition': str})
    required = {'participant', 'condition', 'stage2_beta'}
    if not required.issubset(values.columns):
        raise ValueError(f'{values_path} is missing columns: {sorted(required.difference(values.columns))}')
    values['stage2_beta'] = pd.to_numeric(values['stage2_beta'], errors='raise')
    values['condition'] = values['condition'].str.lower()
    counts = values.groupby('participant')['condition'].nunique()
    complete = counts[counts == 4].index
    values = values.loc[values['participant'].isin(complete)].copy()
    values['condition'] = pd.Categorical(values['condition'], CONDITION_ORDER, ordered=True)
    return values, corrp_img, roi

def plot_condition_bars(values, title):
    summary = values.groupby('condition', observed=False)['stage2_beta'].agg(['mean', 'sem', 'count']).reindex(CONDITION_ORDER)
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    colors = ['#8A8F98', '#2878B5', '#D95F59', '#3A9D6F']
    positions = np.arange(len(CONDITION_ORDER))
    ax.bar(positions, summary['mean'], yerr=summary['sem'], capsize=5, color=colors, edgecolor='#222222', linewidth=0.8, alpha=0.86)
    for position, condition in zip(positions, CONDITION_ORDER):
        points = values.loc[values['condition'] == condition, 'stage2_beta'].sort_values().to_numpy()
        jitter = np.linspace(-0.12, 0.12, len(points))
        ax.scatter(position + jitter, points, s=17, color='#202020', alpha=0.48, linewidth=0, zorder=3)
    ax.set_xticks(positions, CONDITION_LABELS)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_ylabel('Dual-regression stage-2 beta\n(mean ± SEM)')
    ax.set_title(title, fontsize=12, pad=12)
    ax.text(0.99, 0.02, f"n = {int(summary['count'].min())} participants", transform=ax.transAxes, ha='right', va='bottom', color='#555555', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return summary

## 4. All primary and secondary significant results

Each warm-colored overlay contains only voxels with cluster-extent FWE-corrected `1-p > 0.95` (corrected `p < 0.05`). The overlay encodes corrected significance, not connectivity magnitude. Use the NiiVue controls to change slice position, orientation, opacity, and rendering mode.

In [ ]:
CACHE_DIR = PROJECT_ROOT / 'derivatives' / 'fsl' / 'randomise_summary' / '.notebook_cache'
CACHE_DIR.mkdir(exist_ok=True)
mni_img = datasets.load_mni152_template(resolution=2)
mni_path = Path(mni_img.get_filename()) if mni_img.get_filename() else CACHE_DIR / 'MNI152_T1_2mm.nii.gz'
if not mni_path.is_file():
    nib.save(mni_img, mni_path)

def thresholded_map_path(row, corrp_img, roi):
    source = result_image(row)
    destination = CACHE_DIR / source.name.replace('_stat-corrp_statmap.nii.gz', '_desc-thresholded_statmap.nii.gz')
    thresholded = image.new_img_like(corrp_img, np.where(roi, np.asarray(corrp_img.dataobj), 0.0), copy_header=True)
    nib.save(thresholded, destination)
    return destination

for result_number, (row_index, row) in enumerate(significant.iterrows(), start=1):
    values, corrp_img, roi = extract_condition_values(row_index)
    sidecar = result_sidecar(row)
    corrected_p = 1.0 - float(row['peak_corrp'])
    n_participants = values['participant'].nunique()
    map_label = f"map {int(row['component'])}" if row['analysis'] == 'smith09' else f"component {int(row['component'])}"
    display(Markdown(
        f"### Result {result_number}: {result_label(row)}\n\n"
        f"- **Analysis family:** {str(row['family']).title()}.\n"
        f"- **Network definition:** {row['analysis_description']}, {map_label}.\n"
        f"- **Tested contrast:** `{row['condition_contrast']}`, {row['design_contrast']} ({row['direction']}); this is **{row['effect']}**.\n"
        f"- **Inference:** one-sample randomise, {int(sidecar.get('NumberOfPermutations', 5000)):,} permutations, cluster-forming *t* = {float(sidecar.get('ClusterFormingTThreshold', 3.1)):.1f}.\n"
        f"- **Corrected result:** peak FWE-corrected *p* = {corrected_p:.4f}; {int(roi.sum())} suprathreshold voxels.\n"
        f"- **Bar plot:** mean stage-2 beta ± SEM across {n_participants} participants within the displayed cluster."
    ))
    overlay_path = thresholded_map_path(row, corrp_img, roi)
    viewer = NiiVue()
    viewer.load_volumes([
        {'path': str(mni_path), 'name': 'MNI152 T1 2 mm', 'colormap': 'gray', 'opacity': 1.0},
        {'path': str(overlay_path), 'name': row['effect'], 'colormap': 'red', 'opacity': 0.80, 'cal_min': CORRP_THRESHOLD, 'cal_max': 1.0},
    ])
    display(viewer)
    plot_condition_bars(values, result_label(row))

## 5. Nonredundant within-subject data-quality summaries

The N=27 analysis contains participants with one usable run in each condition. Starting from 31 participants, `sub-216` and `sub-232` have no task-rest BOLD, `sub-212` has only SHAM and VLPFC runs, and `sub-233` has four BOLD runs but zero-row placeholder events files with no condition values. No participant was excluded from this N=27 analysis for motion or tSNR.

The QC analysis separates group-level condition pattern from participant exclusion. Subject-centered signed deviations and three orthogonal contrasts retain direction without seven correlated pairwise screens. For each QC metric, the omnibus statistic is the sum of the squared group means for the four centered conditions. The permutation test independently shuffles the four condition labels within every participant 20,000 times; its p-value is the proportion of shuffled statistics at least as large as observed. It asks whether condition labels explain a consistent group pattern, not whether any participant should be excluded.

The exclusion rule uses the lab's usual Tukey boxplot criterion. For each participant and metric, the SD across SHAM, RTPJ, VLPFC, and BOTH summarizes condition-level spread. An SD above `Q3 + 1.5 × IQR` is one metric-specific flag. Exclusion requires flags for all three prespecified metrics: tSNR, mean FD, and percentage of volumes above 0.20-mm FD.

In [ ]:
from matplotlib.lines import Line2D

QC_DIR = PROJECT_ROOT / 'derivatives' / 'qc'
qc_runs = pd.read_csv(QC_DIR / 'task-rest_mriqc_outliers.tsv', sep='\t')
QC_METRICS = {
    'tsnr': ('tSNR', 'tSNR units'),
    'fd_mean': ('Mean FD', 'FD (mm)'),
    'fd_perc': ('Volumes with FD > 0.20 mm', 'Volumes (%)'),
}
condition_sets = qc_runs.groupby('participant')['condition'].agg(lambda values: set(values.dropna()))
analysis_participants = sorted(
    condition_sets[condition_sets.apply(lambda values: values == set(CONDITION_ORDER))].index
)
qc_runs = qc_runs.loc[qc_runs['participant'].isin(analysis_participants)].copy()
qc_pivots = {
    metric: qc_runs.pivot(index='participant', columns='condition', values=metric).loc[
        analysis_participants, CONDITION_ORDER
    ]
    for metric in QC_METRICS
}
if any(table.isna().any().any() for table in qc_pivots.values()):
    raise ValueError('The N=27 QC table is missing a participant-condition value.')

qc_centered = {
    metric: table.subtract(table.mean(axis=1), axis=0)
    for metric, table in qc_pivots.items()
}

ORTHOGONAL_CONTRASTS = {
    'Active mean − SHAM': np.array([-1.0, 1 / 3, 1 / 3, 1 / 3]),
    'BOTH − single-site mean': np.array([0.0, -0.5, -0.5, 1.0]),
    'RTPJ − VLPFC': np.array([0.0, 1.0, -1.0, 0.0]),
}
qc_orthogonal = {
    metric: pd.DataFrame(
        {label: table.to_numpy() @ weights for label, weights in ORTHOGONAL_CONTRASTS.items()},
        index=table.index,
    )
    for metric, table in qc_pivots.items()
}

def condition_permutation_test(centered, n_permutations=20000, seed=20260623):
    values = centered.to_numpy()
    observed = float(np.square(values.mean(axis=0)).sum())
    rng = np.random.default_rng(seed)
    exceedances = 0
    for _ in range(n_permutations):
        orders = np.argsort(rng.random(values.shape), axis=1)
        permuted = np.take_along_axis(values, orders, axis=1)
        exceedances += float(np.square(permuted.mean(axis=0)).sum()) >= observed
    return observed, (exceedances + 1) / (n_permutations + 1)

omnibus = {
    metric: condition_permutation_test(table, seed=20260623 + index)
    for index, (metric, table) in enumerate(qc_centered.items())
}

mean_marker = {
    'marker': 'D', 'markersize': 5, 'markerfacecolor': '#B42318',
    'markeredgecolor': '#B42318', 'ecolor': '#B42318', 'elinewidth': 1.3,
    'capsize': 4, 'linestyle': 'none', 'zorder': 5,
}

fig, axes = plt.subplots(3, 1, figsize=(11.5, 13), sharex=True)
for axis, (metric, (label, unit)) in zip(axes, QC_METRICS.items()):
    table = qc_centered[metric]
    distributions = [table[condition].to_numpy() for condition in CONDITION_ORDER]
    boxes = axis.boxplot(
        distributions, positions=range(1, 5), widths=0.56, whis=1.5,
        showfliers=False, patch_artist=True,
        medianprops={'color': '#111827', 'linewidth': 1.4},
    )
    for box in boxes['boxes']:
        box.set(facecolor='#DDE3EA', edgecolor='#4B5563', alpha=0.72)
    for position, values in enumerate(distributions, start=1):
        jitter = np.linspace(-0.16, 0.16, len(values))
        axis.scatter(position + jitter, values, s=22, color='#64748B', alpha=0.68, linewidth=0)
        axis.errorbar(
            position, values.mean(), yerr=values.std(ddof=1) / np.sqrt(len(values)),
            **mean_marker,
        )
    axis.axhline(0, color='#111827', linewidth=0.9)
    axis.set_ylabel(f'Condition − subject mean\n({unit})')
    axis.set_title(f'{label}: omnibus within-subject permutation p = {omnibus[metric][1]:.3f}', fontsize=12)
    axis.grid(axis='y', color='#D1D5DB', linewidth=0.7, alpha=0.65)
    axis.spines[['top', 'right']].set_visible(False)
axes[-1].set_xticks(range(1, 5), CONDITION_LABELS)
fig.suptitle('Signed condition deviations within each participant', fontsize=15, y=0.995)
fig.legend(
    handles=[
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#64748B', markeredgecolor='none', label='Participant'),
        Line2D([0], [0], marker='D', color='#B42318', markerfacecolor='#B42318', linestyle='none', label='Group mean ± SEM'),
    ],
    loc='upper center', bbox_to_anchor=(0.5, 0.969), ncol=2, frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)

fig, axes = plt.subplots(3, 1, figsize=(11.5, 13), sharex=True)
contrast_rows = []
for axis, (metric, (label, unit)) in zip(axes, QC_METRICS.items()):
    table = qc_orthogonal[metric]
    distributions = [table[column].to_numpy() for column in ORTHOGONAL_CONTRASTS]
    boxes = axis.boxplot(
        distributions, positions=range(1, 4), widths=0.56, whis=1.5,
        showfliers=False, patch_artist=True,
        medianprops={'color': '#111827', 'linewidth': 1.4},
    )
    for box in boxes['boxes']:
        box.set(facecolor='#DDE3EA', edgecolor='#4B5563', alpha=0.72)
    for position, (contrast, values) in enumerate(zip(ORTHOGONAL_CONTRASTS, distributions), start=1):
        jitter = np.linspace(-0.16, 0.16, len(values))
        axis.scatter(position + jitter, values, s=22, color='#64748B', alpha=0.68, linewidth=0)
        axis.errorbar(
            position, values.mean(), yerr=values.std(ddof=1) / np.sqrt(len(values)),
            **mean_marker,
        )
        contrast_rows.append({
            'Metric': label,
            'Orthogonal contrast': contrast,
            'Mean': values.mean(),
            'SEM': values.std(ddof=1) / np.sqrt(len(values)),
            'Positive N': int((values > 0).sum()),
            'Negative N': int((values < 0).sum()),
        })
    axis.axhline(0, color='#111827', linewidth=0.9)
    axis.set_ylabel(unit)
    axis.set_title(label, fontsize=12)
    axis.grid(axis='y', color='#D1D5DB', linewidth=0.7, alpha=0.65)
    axis.spines[['top', 'right']].set_visible(False)
axes[-1].set_xticks(range(1, 4), list(ORTHOGONAL_CONTRASTS), rotation=15, ha='right')
fig.suptitle('Three nonredundant signed contrasts span all condition differences', fontsize=15, y=0.995)
fig.legend(
    handles=[
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#64748B', markeredgecolor='none', label='Participant'),
        Line2D([0], [0], marker='D', color='#B42318', markerfacecolor='#B42318', linestyle='none', label='Group mean ± SEM'),
    ],
    loc='upper center', bbox_to_anchor=(0.5, 0.969), ncol=2, frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)
display(pd.DataFrame(contrast_rows).style.format({'Mean': '{:.4f}', 'SEM': '{:.4f}'}).hide(axis='index'))

def tukey_upper(values):
    q1, q3 = np.quantile(values, [0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)

spread = pd.DataFrame(index=analysis_participants)
spread_flags = pd.DataFrame(index=analysis_participants)
for metric, table in qc_pivots.items():
    spread[f'{metric}_sd'] = table.std(axis=1, ddof=1)
    spread[f'{metric}_range'] = table.max(axis=1) - table.min(axis=1)
    spread[f'{metric}_highest_condition'] = table.idxmax(axis=1)
    spread[f'{metric}_lowest_condition'] = table.idxmin(axis=1)
    fence = tukey_upper(spread[f'{metric}_sd'])
    spread_flags[metric] = spread[f'{metric}_sd'] > fence

fig, axes = plt.subplots(1, 3, figsize=(14, 5.4))
for axis, (metric, (label, unit)) in zip(axes, QC_METRICS.items()):
    values = spread[f'{metric}_sd'].sort_values()
    fence = tukey_upper(values)
    boxes = axis.boxplot(
        [values.to_numpy()], positions=[1], widths=0.5, whis=1.5,
        showfliers=False, patch_artist=True,
        medianprops={'color': '#111827', 'linewidth': 1.4},
    )
    boxes['boxes'][0].set(facecolor='#DDE3EA', edgecolor='#4B5563', alpha=0.72)
    jitter = np.linspace(-0.18, 0.18, len(values))
    flagged = values > fence
    colors = np.where(flagged, '#C2410C', '#64748B')
    axis.scatter(1 + jitter, values, c=colors, s=27, alpha=0.85, linewidth=0, zorder=3)
    axis.scatter(1, fence, marker='_', s=280, color='#B91C1C', linewidth=2.0, zorder=4)
    for x, value, participant, is_flagged in zip(1 + jitter, values, values.index, flagged):
        if is_flagged:
            axis.annotate(
                participant.replace('sub-', ''), (x, value), xytext=(0, 7),
                textcoords='offset points', ha='center', fontsize=8, color='#9A3412',
            )
    axis.set_xticks([1], [label])
    axis.set_ylabel(f'Within-subject SD ({unit})')
    axis.set_title(f'Upper fence = {fence:.3g}', fontsize=11)
    axis.grid(axis='y', color='#D1D5DB', linewidth=0.7, alpha=0.65)
    axis.spines[['top', 'right']].set_visible(False)
fig.suptitle('Participant-level magnitude across all four conditions', fontsize=15, y=0.99)
fig.legend(
    handles=[
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#64748B', markeredgecolor='none', label='Within fence'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#C2410C', markeredgecolor='none', label='Spread outlier'),
        Line2D([0], [0], marker='_', color='#B91C1C', markersize=15, linewidth=0, label='Tukey upper fence'),
    ],
    loc='upper center', bbox_to_anchor=(0.5, 0.93), ncol=3, frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.87))
display(fig)
plt.close(fig)

spread['n_metric_outliers'] = spread_flags.sum(axis=1)
spread['decision'] = np.where(spread['n_metric_outliers'] == len(QC_METRICS), 'exclude', 'include')
review = spread.loc[spread['n_metric_outliers'] > 0, [
    'tsnr_sd', 'fd_mean_sd', 'fd_perc_sd',
    'tsnr_highest_condition', 'tsnr_lowest_condition',
    'fd_mean_highest_condition', 'fd_mean_lowest_condition',
    'fd_perc_highest_condition', 'fd_perc_lowest_condition',
    'n_metric_outliers', 'decision',
]].copy()
review.insert(0, 'Participant', review.index)
review = review.sort_values(['n_metric_outliers', 'Participant'], ascending=[False, True])
display(review.style.format({
    'tsnr_sd': '{:.3f}', 'fd_mean_sd': '{:.4f}', 'fd_perc_sd': '{:.3f}',
}).hide(axis='index'))

excluded = spread.index[spread['decision'] == 'exclude'].tolist()
omnibus_text = ', '.join(
    f"{QC_METRICS[metric][0]} p={result[1]:.3f}" for metric, result in omnibus.items()
)
display(Markdown(
    '### Reading these diagnostics\n\n'
    f'- The omnibus within-subject permutation tests are **{omnibus_text}**. None suggests that condition labels organize a consistent group-level QC pattern. These are not participant-exclusion tests.\n'
    '- The centered condition plots retain direction: values above zero are above that participant’s own four-condition mean. For motion, positive means worse; for tSNR, positive means better.\n'
    '- The three orthogonal contrasts are nonredundant and jointly encode every possible condition difference, avoiding seven correlated pairwise screens.\n'
    '- The participant exclusion rule is the usual upper Tukey fence applied separately to four-condition SD for tSNR, mean FD, and high-motion percentage. A participant must be an outlier for **all three metrics**.\n'
    f"- Participants meeting all three metric-specific boxplot criteria: **{', '.join(excluded) or 'none'}**. Therefore the QC rule excludes nobody from the current N=27 analysis.\n"
    '- A spread flag says that a participant varies unusually across conditions. The highest/lowest-condition columns show direction, while the group plots show whether those directions align across people.\n'
    '- Participants flagged on only one or two metrics remain included. The table preserves their magnitudes and directions for transparent review.'
))


## 6. Interpretation notes

The bars and participant points show values extracted from the stage-2 beta maps. Because each displayed region was selected from the same group contrast, these values are a descriptive visualization of the result, not an independent ROI test. Cluster-extent corrected p-values control spatial inference within each randomise job. Across jobs, each reported network/contrast is treated as an individual hypothesis rather than as evidence for the disjunctive claim that at least one effect exists somewhere. Under Rubin's inference-based account, that distinction determines whether an across-job alpha adjustment is relevant. Confirmatory ROI inference requires an independently defined mask or held-out data.